# Preparing a Model for Deployment: The Full Save/Load/Predict Cycle

Deployment is not just saving a model file. A real deployment artifact is a bundle: the trained pipeline, the preprocessing transformations, and a metadata file describing what the model is and what it expects. This notebook builds that bundle from scratch and verifies it works correctly after loading.

**Learning objectives**
1. Understand what files a deployment artifact must contain beyond just the model weights.
2. Build a `sklearn.Pipeline` that combines preprocessing and a classifier in one object.
3. Save the pipeline with joblib, delete it from memory, reload it, and verify predictions are identical.
4. Save a metadata JSON alongside the model and write a `load_model_bundle()` function that validates it.

## 🔗 Where this fits

**Builds on:** Course 04 (AIAT 114) — Unit 1, lesson 03 "03. Data Preprocessing" — the `StandardScaler` and `OneHotEncoder` you fitted on training data have to travel with the model; a deployment that reloads only the estimator transforms production inputs differently and fails silently.


## 📰 $327 million lost to a missing unit label

On **23 September 1999** NASA lost the **Mars Climate Orbiter**. The spacecraft was healthy, the navigation mathematics was correct, and the software on both sides ran exactly as written. The problem was the interface between them: the ground software supplied thruster impulse in **pound-force seconds**, while the navigation software expected **newton-seconds** — a factor of about 4.45. The orbiter arrived at roughly 57 km altitude instead of the planned 226 km and was destroyed in the Martian atmosphere. The Mishap Investigation Board traced the loss to that unit specification in a file the two teams shared. The mission cost about **$327.6 million**.

Nothing in this notebook is about spacecraft. Everything in it is about the same failure mode: **a numeric array is meaningless without the description of what its columns are and what units they are in**, and a model file alone carries no such description. `joblib.load()` hands you an object that will happily predict from a scrambled or mis-scaled row and return a confident answer.

**What goes wrong without this lesson.** The single most common ML deployment bug is shipping the estimator without the fitted preprocessing — the scaler stays in the training notebook, the server feeds raw features straight into a model trained on standardized ones, and *nothing errors*. Accuracy quietly collapses while every dashboard stays green. Bundling the scaler inside the pipeline makes that bug impossible to write; the metadata file makes the remaining assumptions inspectable six months later.

## 1  What does a deployment artifact need?

A model in production needs three things:

| File | Purpose |
|------|--------|
| `pipeline.joblib` | The trained model + all preprocessing steps |
| `metadata.json` | Version, feature names, accuracy, training date |
| `requirements.txt` | Exact package versions the model was trained with |

If you save the model but not the scaler, predictions will be wrong. If you save the model but not the metadata, you have no way to check whether the file on the server is the one you intended to deploy.


In [1]:
# WHAT: load iris, keep the human-readable feature/class names, and split the data.
# WHY: a deployable model needs more than weights — the names captured here become
# part of the model's metadata so the serving side can label inputs and outputs.
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np
import joblib
import json
import os
from datetime import datetime

iris = load_iris()
FEATURE_NAMES = list(iris.feature_names)
CLASS_NAMES = list(iris.target_names)

# Hold out 20% so the accuracy stored in metadata is an honest holdout number.
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Print what will go into metadata — students should recognize these later.
print("Feature names:", FEATURE_NAMES)
print("Class names  :", CLASS_NAMES)
print(f"Training samples: {len(X_train)}   Test samples: {len(X_test)}")

Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Class names  : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Training samples: 120   Test samples: 30


## 2  Build a preprocessing pipeline

Wrapping the scaler and classifier in a `Pipeline` means you only have to call `pipeline.predict(raw_features)` — the pipeline applies the scaler automatically. This is what you should always save: the whole pipeline, not just the classifier.


In [2]:
# WHAT: bundle preprocessing + model into ONE sklearn Pipeline and train it.
# WHY: if the scaler is saved separately from the model, serving code can forget it —
# the classic train/serve skew bug. One pipeline object makes that impossible.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fitting the pipeline fits the scaler AND the forest in the right order.
pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

# Reference predictions — we'll verify the loaded pipeline produces the same values
reference_preds = pipeline.predict(X_test)

# The scaler mean proves preprocessing parameters live inside the artifact.
print(f"Pipeline accuracy: {accuracy:.2%}")
print(f"Steps in pipeline: {[s[0] for s in pipeline.steps]}")
print(f"Scaler mean (first feature): {pipeline.named_steps['scaler'].mean_[0]:.4f}")

Pipeline accuracy: 100.00%
Steps in pipeline: ['scaler', 'classifier']
Scaler mean (first feature): 5.8092


## 3  Save the pipeline with joblib


In [3]:
# WHAT: save the whole pipeline into a dedicated bundle directory.
# WHY: one folder = one deployable unit; the metadata file will sit next to the
# model so they can never be shipped separately.
BUNDLE_DIR = "/tmp/iris_model_bundle"
os.makedirs(BUNDLE_DIR, exist_ok=True)

PIPELINE_PATH = os.path.join(BUNDLE_DIR, "pipeline.joblib")
joblib.dump(pipeline, PIPELINE_PATH)

# Report the artifact size — deployment systems care about download/startup cost.
size_kb = os.path.getsize(PIPELINE_PATH) / 1024
print(f"Pipeline saved to: {PIPELINE_PATH}")
print(f"File size: {size_kb:.1f} KB")

Pipeline saved to: /tmp/iris_model_bundle/pipeline.joblib
File size: 183.1 KB


## 4  Load fresh and verify identical predictions

We delete the original object from memory (simulating a fresh server), reload from disk, and confirm predictions are bit-for-bit identical to the reference.


In [4]:
# Delete the original pipeline to simulate a fresh environment
del pipeline

# Reload from disk
loaded_pipeline = joblib.load(PIPELINE_PATH)

loaded_preds = loaded_pipeline.predict(X_test)
match = np.array_equal(loaded_preds, reference_preds)

print(f"Predictions match original: {match}")
print(f"Loaded pipeline accuracy  : {loaded_pipeline.score(X_test, y_test):.2%}")
print(f"Scaler mean still intact  : {loaded_pipeline.named_steps['scaler'].mean_[0]:.4f}")

# Confirm the scaler is embedded — raw features go in, correct predictions come out
sample_raw = X_test[:3]
print(f"\nSample predictions on raw features (scaler applied inside pipeline):")
for i, pred in enumerate(loaded_pipeline.predict(sample_raw)):
    print(f"  Sample {i}: {CLASS_NAMES[pred]}")


Predictions match original: True
Loaded pipeline accuracy  : 100.00%
Scaler mean still intact  : 5.8092

Sample predictions on raw features (scaler applied inside pipeline):
  Sample 0: versicolor
  Sample 1: setosa
  Sample 2: virginica


## 5  Save model metadata as JSON

The metadata file records what this model is, what it was trained on, and how well it performed. Without this file, you cannot tell which version of the model is running in production.


In [5]:
# WHAT: write a metadata.json next to the model with version, date, accuracy, schema.
# WHY: six months from now nobody remembers what 'pipeline.joblib' is — metadata
# makes the artifact self-describing, auditable, and safe to roll back.
import sklearn

# Everything a future engineer (or an automated gate) needs to trust this model.
metadata = {
    "model_name": "iris-classifier",
    "version": "1.0.0",
    "training_date": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "feature_names": FEATURE_NAMES,
    "class_names": list(CLASS_NAMES),
    "n_features": len(FEATURE_NAMES),
    "n_classes": len(CLASS_NAMES),
    "test_accuracy": round(loaded_pipeline.score(X_test, y_test), 4),
    "sklearn_version": sklearn.__version__,
    "pipeline_steps": [s[0] for s in loaded_pipeline.steps],
}

# JSON, not pickle: metadata must be readable without Python or sklearn.
METADATA_PATH = os.path.join(BUNDLE_DIR, "metadata.json")
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved:")
print(json.dumps(metadata, indent=2))

Metadata saved:
{
  "model_name": "iris-classifier",
  "version": "1.0.0",
  "training_date": "2026-08-23T16:52:11Z",
  "feature_names": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ],
  "class_names": [
    "setosa",
    "versicolor",
    "virginica"
  ],
  "n_features": 4,
  "n_classes": 3,
  "test_accuracy": 1.0,
  "sklearn_version": "1.8.0",
  "pipeline_steps": [
    "scaler",
    "classifier"
  ]
}


/var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/ipykernel_16304/2660466530.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "training_date": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),


## 6  Load the full model bundle

A deployment system should load both files together and validate that the metadata is consistent before serving any traffic.


In [6]:
# WHAT: a loader that refuses to return a model unless the bundle is complete and
# consistent (files exist, feature counts agree between metadata and model).
# WHY: failing loudly at load time is far cheaper than serving wrong predictions —
# this is the receiving end of the packaging contract we just wrote.
def load_model_bundle(bundle_dir):
    """Load a model pipeline and its metadata, with basic validation."""
    pipeline_path = os.path.join(bundle_dir, "pipeline.joblib")
    metadata_path = os.path.join(bundle_dir, "metadata.json")

    if not os.path.exists(pipeline_path):
        raise FileNotFoundError(f"pipeline.joblib not found in {bundle_dir}")
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"metadata.json not found in {bundle_dir}")

    model = joblib.load(pipeline_path)
    with open(metadata_path) as f:
        meta = json.load(f)

    # Validate that the model expects the right number of features
    expected_features = meta["n_features"]
    scaler = model.named_steps["scaler"]
    actual_features = scaler.n_features_in_
    if expected_features != actual_features:
        raise ValueError(
            f"Metadata says {expected_features} features, "
            f"but model has {actual_features}"
        )

    return model, meta


# Load the bundle
model, meta = load_model_bundle(BUNDLE_DIR)

print(f"Loaded model    : {meta['model_name']} v{meta['version']}")
print(f"Trained on      : {meta['training_date']}")
print(f"Feature names   : {meta['feature_names']}")
print(f"Test accuracy   : {meta['test_accuracy']:.2%}")

# Confirm predictions still match
# Final proof: the loaded bundle predicts exactly like the pipeline we trained.
bundle_preds = model.predict(X_test)
print(f"\nPredictions match reference: {np.array_equal(bundle_preds, reference_preds)}")

Loaded model    : iris-classifier v1.0.0
Trained on      : 2026-08-23T16:52:11Z
Feature names   : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Test accuracy   : 100.00%

Predictions match reference: True


## 💬 Discuss

The reload check printed `Predictions match original: True` and `Scaler mean still intact: 5.8092` — the same number as before the save. That is the comparison this notebook rests on: **save → delete from memory → reload → identical predictions**. Now push on what it does and does not prove.

1. `np.array_equal(loaded_preds, reference_preds)` compares *class labels*. Would it still print `True` if the reloaded scaler had subtly different floating-point internals, or if `predict_proba` had shifted by 0.01 across the board? What would you compare instead, and is the stricter check worth its false alarms?
2. Our `metadata.json` records `sklearn_version` but the loader never checks it. Should `load_model_bundle()` *refuse* to load a bundle built by a different scikit-learn version, *warn*, or *ignore* it? Argue it as the person on call at 2 a.m., then argue it again as the person trying to ship a hotfix.
3. The metadata stores `test_accuracy: 1.0`. Six months from now, that number is the only evidence anyone has about this model's quality. What would you add to the file so that a future colleague can tell whether 100% is impressive or a warning sign?

## Summary

A deployment artifact is a bundle, not just a model file. The three required components are:

| Component | Why it matters |
|---|---|
| `pipeline.joblib` (with scaler embedded) | The model and all preprocessing in one object — raw input goes in, prediction comes out |
| `metadata.json` | Records version, features, accuracy — makes deployments auditable |
| Verification step | Confirms the loaded model gives the same predictions as the original |

Saving the classifier without the scaler is a common mistake that causes incorrect predictions in production because the input distribution at inference time won't match what the model was trained on.


## Self-check

1. **What goes wrong if you save only the classifier but not the scaler?** Run `model.predict(X_test)` after removing the scaler step and observe the difference in predictions.
2. **What should you store in the metadata JSON?** Look at the metadata dict in Section 5 — which fields would you need to debug a model behaving unexpectedly in production?
3. **How would you check that the loaded model gives the same predictions as the original?** Look at the `np.array_equal(loaded_preds, reference_preds)` check in Section 4 — what would a `False` result tell you?


## ⚠️ Where this breaks

- **A hash-free bundle is an honour system.** Nothing here binds `pipeline.joblib` to `metadata.json`. Copy a newer model into the folder and the loader will keep reporting the old version, the old accuracy and the old feature names — with complete confidence. Unit 5 notebook 06 adds the data hash and git commit that close this gap; until then, treat the metadata as documentation, not as verification.
- **`Pipeline` protects you from *forgetting* preprocessing, not from *changing* it.** If the production feature pipeline upstream starts computing `mean area` differently, the model receives numbers that are correctly *shaped* and wrong in *meaning* — the Mars Climate Orbiter failure, in tabular form. No amount of bundling detects that. Only comparing live input distributions against the training baseline does (Unit 5, notebook 04).
- **The assumption that must hold:** the environment that loads the bundle can reconstruct every class inside it. A `joblib` file stores references, not code. Custom transformers, lambdas, and anything defined in a notebook cell will fail to unpickle elsewhere — often with an unhelpful `AttributeError`. If your pipeline contains custom steps, they must live in an importable module that ships with the service.
- **`utcnow()` and other silent metadata rot.** The cell above emits a `DeprecationWarning` for `datetime.utcnow()`, and the timestamp it writes has no timezone. In an audit trail, an ambiguous timestamp is close to no timestamp. Metadata is only as trustworthy as its least careful field.
- **The cheaper alternative.** If you are already using MLflow or another registry (Unit 5, notebook 05), it stores the artifact, the parameters, the metrics and the environment for you, and a hand-rolled `metadata.json` becomes a second source of truth that will drift from the first. Hand-roll this only when you genuinely have no registry.

## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Olston, C., Fiedel, N., Gorovoy, K., et al. (2017). *TensorFlow-Serving: Flexible, High-Performance ML Serving*. NeurIPS Workshop on ML Systems. <https://arxiv.org/abs/1712.06139>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access. <https://arxiv.org/abs/2205.02302>
